In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!nvidia-smi

In [ ]:
!pip install mediapipe opencv-python-headless -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 122.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 16.3 MB/s eta 0:00:00


In [ ]:
import zipfile, os

print("Unzipping alphabet... (takes 2-3 mins)")
with zipfile.ZipFile('/content/drive/MyDrive/SignSpeak/asl_alphabet_train.zip', 'r') as z:
    z.extractall('/content/asl_alphabet')

print("Unzipping numbers... (takes 1 min)")
with zipfile.ZipFile('/content/drive/MyDrive/SignSpeak/Train_Nums.zip', 'r') as z:
    z.extractall('/content/asl_numbers')

print("Done! Checking folders...")
print(os.listdir('/content/asl_alphabet'))
print(os.listdir('/content/asl_numbers'))

Unzipping alphabet... (takes 2-3 mins)
Unzipping numbers... (takes 1 min)
Done! Checking folders...
['asl_alphabet_train']
['10', '9', '7', '8', '1', '2', '4', '3', '5', '6', 'Blank']


In [ ]:
import os
print(os.listdir('/content/asl_alphabet/asl_alphabet_train'))
print(os.listdir('/content/asl_numbers/1'))  # check what's inside a number folder

['nothing', 'M', 'A', 'O', 'W', 'P', 'J', 'Y', 'I', 'H', 'C', 'V', 'U', 'E', 'T', 'B', 'S', 'del', 'L', 'N', 'F', 'G', 'X', 'Q', 'space', 'Z', 'D', 'R', 'K']
['035cd08f-1bad-4a8f-82df-451dc05769ea.rgb_0000.png', 'd56ced97-0744-447c-a583-6c0faafdd353.rgb_0000.png', 'c993d54e-9a2b-4765-842d-1f5805933c73.rgb_0000.png', '284c89d7-b6f3-48de-9ba4-33f1531d634c.rgb_0000.png', '02efa345-f927-426a-8e49-6efd0fece4b6.rgb_0000.png', '8dbc413a-b53d-4bed-afa5-766ff6e60bc8.rgb_0000.png', '9d46996c-0965-496c-a328-4f3bfa97cbaa.rgb_0000.png', '5953fafe-1e0b-4793-8267-288c0b0086fd.rgb_0000.png', '76cabe3c-f9b5-47cb-adc2-a81d1d047f8c.rgb_0000.png', 'a5e979cf-8bdf-4446-a0be-0b40dfe7b486.rgb_0000.png', 'daf66d03-7857-4f4c-afc2-f4a4879d9258.rgb_0000.png', '9696835b-756a-47e0-8aae-276831a89149.rgb_0000.png', 'd0f8464f-4789-4b44-b746-300ef41c65a7.rgb_0000.png', '810ea99b-1a25-4154-8720-41b596281a49.rgb_0000.png', 'fe094734-2223-4082-bbf5-d9c490166151.rgb_0000.png', '73a5c483-9ea6-4808-b10c-bfaf42d955a5.rgb_0000

In [ ]:
import mediapipe as mp
print(mp.__version__)

0.10.33


In [ ]:
import cv2
import numpy as np
import mediapipe as mp
import pandas as pd
import json
import os
import urllib.request
from pathlib import Path
from tqdm import tqdm

SAVE_DIR          = '/content/drive/MyDrive/SignSpeak'
ASL_ALPHABET_PATH = '/content/asl_alphabet/asl_alphabet_train'
ASL_NUMBERS_PATH  = '/content/asl_numbers'
MODEL_PATH        = '/content/hand_landmarker.task'

os.makedirs(SAVE_DIR, exist_ok=True)

# ── Download model if needed ──
if not os.path.exists(MODEL_PATH):
    print("Downloading hand_landmarker.task model...")
    urllib.request.urlretrieve(
        'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task',
        MODEL_PATH
    )
    print("Done.")

# ── Tasks API ──
options = mp.tasks.vision.HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=mp.tasks.vision.RunningMode.IMAGE,
    num_hands=1,
    min_hand_detection_confidence=0.4
)

def extract_landmarks_from_image(image_path, detector):
    img = cv2.imread(str(image_path))
    if img is None:
        return None
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)
    result = detector.detect(mp_image)
    if not result.hand_landmarks:
        return None
    coords = np.array(
        [[lm.x, lm.y, lm.z] for lm in result.hand_landmarks[0]],
        dtype=np.float32
    )
    coords -= coords[0]
    scale = np.linalg.norm(coords[9])
    if scale > 1e-6:
        coords /= scale
    return coords.flatten()

# ── Label map: A-Z (0-25), 1-10 (26-35), Blank (36) ──
LABEL_MAP = {
    **{chr(i): idx for idx, i in enumerate(range(ord('A'), ord('Z') + 1))},  # A-Z → 0-25
    '1': 26, '2': 27, '3': 28, '4': 29, '5': 30,                            # 1-5 → 26-30
    '6': 31, '7': 32, '8': 33, '9': 34,                                      # 6-9 → 31-34
    '10': 35,                                                                  # 10  → 35
    'Blank': 36                                                                # Blank → 36
}

with open(f'{SAVE_DIR}/label_map.json', 'w') as f:
    json.dump(LABEL_MAP, f, indent=2)
print(f"Label map saved — {len(LABEL_MAP)} classes")
print(json.dumps(LABEL_MAP, indent=2))

# ── Extract ──
records = []
failed  = 0

ALPHABET_CLASSES = [(k, v) for k, v in LABEL_MAP.items() if k.isalpha() and k != 'Blank']
NUMBER_CLASSES   = [(k, v) for k, v in LABEL_MAP.items() if k.isdigit() or k == '10' or k == 'Blank']

with mp.tasks.vision.HandLandmarker.create_from_options(options) as detector:

    # ── Alphabets A-Z ──
    for label_name, label_idx in tqdm(ALPHABET_CLASSES, desc="A-Z"):
        class_dir = Path(ASL_ALPHABET_PATH) / label_name
        if not class_dir.is_dir():
            print(f"Warning: folder not found for '{label_name}'")
            continue
        for img_path in (list(class_dir.glob('*.jpg'))  +
                         list(class_dir.glob('*.jpeg')) +
                         list(class_dir.glob('*.png'))):
            features = extract_landmarks_from_image(img_path, detector)
            if features is not None:
                records.append([label_idx] + features.tolist())
            else:
                failed += 1

    # ── Numbers 1-10 + Blank ──
    for label_name, label_idx in tqdm(NUMBER_CLASSES, desc="1-10 + Blank"):
        class_dir = Path(ASL_NUMBERS_PATH) / label_name
        if not class_dir.is_dir():
            print(f"Warning: folder not found for '{label_name}'")
            continue
        for img_path in (list(class_dir.glob('*.jpg'))  +
                         list(class_dir.glob('*.jpeg')) +
                         list(class_dir.glob('*.png'))):
            features = extract_landmarks_from_image(img_path, detector)
            if features is not None:
                records.append([label_idx] + features.tolist())
            else:
                failed += 1

print(f"\nDone. Samples: {len(records)}, Failed: {failed}")

# ── Save fresh CSV ──
cols = ['label'] + [f'f{i}' for i in range(63)]
df = pd.DataFrame(records, columns=cols)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
df.to_csv(f'{SAVE_DIR}/landmarks_static.csv', index=False)

print(f"\nSaved to {SAVE_DIR}/landmarks_static.csv")
print(f"Shape: {df.shape}")
print(f"Samples per class:\n{df['label'].value_counts().sort_index()}")

Done.
Label map saved — 37 classes
{
  "A": 0,
  "B": 1,
  "C": 2,
  "D": 3,
  "E": 4,
  "F": 5,
  "G": 6,
  "H": 7,
  "I": 8,
  "J": 9,
  "K": 10,
  "L": 11,
  "M": 12,
  "N": 13,
  "O": 14,
  "P": 15,
  "Q": 16,
  "R": 17,
  "S": 18,
  "T": 19,
  "U": 20,
  "V": 21,
  "W": 22,
  "X": 23,
  "Y": 24,
  "Z": 25,
  "1": 26,
  "2": 27,
  "3": 28,
  "4": 29,
  "5": 30,
  "6": 31,
  "7": 32,
  "8": 33,
  "9": 34,
  "10": 35,
  "Blank": 36
}


1-10 + Blank: 100%|██████████| 11/11 [06:42<00:00, 36.56s/it]



Done. Samples: 69552, Failed: 18348

Saved to /content/drive/MyDrive/SignSpeak/landmarks_static.csv
Shape: (69552, 64)
Samples per class:
label
0     2234
1     2222
2     2022
3     2522
4     2331
5     2880
6     2486
7     2412
8     2419
9     2605
10    2719
11    2551
12    1759
13    1371
14    2298
15    2074
16    2159
17    2580
18    2587
19    2382
20    2527
21    2570
22    2473
23    2201
24    2606
25    2388
26     734
27     833
28     866
29     872
30     783
31     802
32     802
33     800
34     816
35     863
36       3
Name: count, dtype: int64


In [ ]:
import pandas as pd

SAVE_DIR = '/content/drive/MyDrive/SignSpeak'

df = pd.read_csv(f'{SAVE_DIR}/landmarks_static.csv')

# Drop Blank (label 36)
df = df[df['label'] != 36].reset_index(drop=True)
df.to_csv(f'{SAVE_DIR}/landmarks_static.csv', index=False)

print(f"Shape after dropping Blank: {df.shape}")
print(f"Samples per class:\n{df['label'].value_counts().sort_index()}")

Shape after dropping Blank: (69549, 64)
Samples per class:
label
0     2234
1     2222
2     2022
3     2522
4     2331
5     2880
6     2486
7     2412
8     2419
9     2605
10    2719
11    2551
12    1759
13    1371
14    2298
15    2074
16    2159
17    2580
18    2587
19    2382
20    2527
21    2570
22    2473
23    2201
24    2606
25    2388
26     734
27     833
28     866
29     872
30     783
31     802
32     802
33     800
34     816
35     863
Name: count, dtype: int64


In [ ]:
import json

with open(f'{SAVE_DIR}/label_map.json') as f:
    label_map = json.load(f)

del label_map['Blank']

with open(f'{SAVE_DIR}/label_map.json', 'w') as f:
    json.dump(label_map, f, indent=2)

print(f"Label map updated — {len(label_map)} classes")

Label map updated — 36 classes


In [ ]:
import pandas as pd
import numpy as np
import json
import os
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

DRIVE_DIR = '/content/drive/MyDrive/SignSpeak'

# ── Load data ──
print("Loading landmarks_static.csv...")
df = pd.read_csv(f'{DRIVE_DIR}/landmarks_static.csv')
print(f"Total: {len(df)} samples, {df['label'].nunique()} classes")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# ── Features and labels ──
X = df[[f'f{i}' for i in range(63)]].values.astype(np.float32)
y = df['label'].values.astype(np.int32)
NUM_CLASSES = int(y.max()) + 1
print(f"Classes: {NUM_CLASSES}")

# ── Split ──
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)
print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

# ── Augmentation ──
def augment_landmarks(sample):
    coords = sample.reshape(21, 3).copy()
    angle = np.random.uniform(-20, 20) * np.pi / 180
    rot = np.array([[np.cos(angle), -np.sin(angle), 0],
                    [np.sin(angle),  np.cos(angle), 0],
                    [0,              0,             1]], dtype=np.float32)
    coords = coords @ rot.T
    coords *= np.random.uniform(0.85, 1.15)
    coords += np.random.normal(0, 0.02, coords.shape).astype(np.float32)
    if np.random.rand() < 0.3:
        coords[:, 0] *= -1
    return coords.flatten()

print("Augmenting training data (5x)...")
X_aug_list = [X_train]
y_aug_list = [y_train]
for _ in range(4):
    X_aug_list.append(np.array([augment_landmarks(x) for x in X_train], dtype=np.float32))
    y_aug_list.append(y_train)

X_train_all = np.concatenate(X_aug_list)
y_train_all = np.concatenate(y_aug_list)
idx = np.random.permutation(len(X_train_all))
X_train_all = X_train_all[idx]
y_train_all = y_train_all[idx]
print(f"Training set: {len(X_train_all)} samples (was {len(X_train)})")

# ── Class weights (handles alphabet vs number imbalance) ──
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))
print("Class weights computed ✅")

# ── Build 1D CNN ──
inputs = keras.Input(shape=(63,))
x = keras.layers.Reshape((21, 3))(inputs)
x = keras.layers.Conv1D(64, 3, padding='same')(x)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.Activation('relu')(x)
x = keras.layers.Conv1D(128, 3, padding='same')(x)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.Activation('relu')(x)
x = keras.layers.Conv1D(256, 3, padding='same')(x)
x = keras.layers.BatchNormalization()(x)
x = keras.layers.Activation('relu')(x)
x = keras.layers.GlobalAveragePooling1D()(x)
x = keras.layers.Dense(256, activation='relu')(x)
x = keras.layers.Dropout(0.4)(x)
x = keras.layers.Dense(128, activation='relu')(x)
x = keras.layers.Dropout(0.3)(x)
outputs = keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = keras.Model(inputs, outputs, name='SignSpeak_1DCNN')
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

# ── Callbacks ──
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=8, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6, verbose=1),
    keras.callbacks.ModelCheckpoint(
        f'{DRIVE_DIR}/best_model.keras', monitor='val_accuracy',
        save_best_only=True, verbose=1),
]

# ── Train ──
print("\nTraining...")
history = model.fit(
    X_train_all, y_train_all,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=256,
    class_weight=class_weight_dict,
    callbacks=callbacks,
    verbose=1
)

# ── Evaluate ──
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Accuracy: {acc*100:.2f}%")

# ── Save ──
model.save(f'{DRIVE_DIR}/signspeak_model.keras')
print(f"Model saved → Google Drive/SignSpeak/signspeak_model.keras")

Loading landmarks_static.csv...
Total: 69549 samples, 36 classes
Classes: 36
Train: 55639 | Val: 6955 | Test: 6955
Augmenting training data (5x)...
Training set: 278195 samples (was 55639)
Class weights computed ✅


Model: "SignSpeak_1DCNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 63)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reshape (Reshape)               │ (None, 21, 3)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 21, 64)         │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 21, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 21, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 21, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 21, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 21, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 21, 256)        │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 21, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 21, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 36)             │         4,644 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 229,028 (894.64 KB)

 Trainable params: 228,132 (891.14 KB)

 Non-trainable params: 896 (3.50 KB)


Training...
Epoch 1/60
1087/1087 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6519 - loss: 1.1214
Epoch 1: val_accuracy improved from None to 0.96046, saving model to /content/drive/MyDrive/SignSpeak/best_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/SignSpeak/best_model.keras
1087/1087 ━━━━━━━━━━━━━━━━━━━━ 21s 11ms/step - accuracy: 0.8295 - loss: 0.5411 - val_accuracy: 0.9605 - val_loss: 0.1196 - learning_rate: 0.0010
Epoch 2/60
1078/1087 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9499 - loss: 0.1677
Epoch 2: val_accuracy improved from 0.96046 to 0.97743, saving model to /content/drive/MyDrive/SignSpeak/best_model.keras

Epoch 2: finished saving model to /content/drive/MyDrive/SignSpeak/best_model.keras
1087/1087 ━━━━━━━━━━━━━━━━━━━━ 5s 5ms/step - accuracy: 0.9541 - loss: 0.1529 - val_accuracy: 0.9774 - val_loss: 0.0787 - learning_rate: 0.0010
Epoch 3/60
1081/1087 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9622 - loss: 0.1254
Epoch 3: val_accuracy im